# Inverse Design for Heat Exchanger Optimization

This notebook performs inverse design to find the optimal lattice cell sizes and inlet velocity for a heat exchanger. The goal is to maximize the surface area while satisfying constraints on mass, pressure drop, and average flow velocity.

The optimization is performed using `scipy.optimize.minimize`.

In [ ]:
import pickle
import numpy as np
import tensorflow as tf
from scipy.optimize import minimize, Bounds, NonlinearConstraint
import time
import pandas as pd

# Load the trained model and scalers
with open('trained_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('scaler_X.pkl', 'rb') as f:
    scaler_X = pickle.load(f)

with open('scaler_y.pkl', 'rb') as f:
    scaler_y = pickle.load(f)

print("Model and scalers loaded successfully.")

## Define Objective and Constraint Functions

The optimization problem is defined as follows:

**Maximize:**
- `Surface Area`

**Subject to:**
- `Mass` < 125 g
- `Pressure Drop` < 8000 Pa
- `Avg Velocity` > 520 mm/s

**With input bounds:**
- 10 mm < `X Cell Size` < 25 mm
- 10 mm < `YZ Cell Size` < 25 mm
- 2500 mm/s < `Inlet Velocity` < 3500 mm/s

We will use `scipy.optimize.minimize`, so we need to define an objective function to *minimize*. We will minimize the negative of the surface area.

In [ ]:
# The order of targets is: ['PressureDrop', 'AvgVelocity', 'Surface Area', 'Mass']
def predict_outputs(x):
    """Helper function to predict outputs for a given input x."""
    # Convert numpy array to dataframe with feature names to avoid warnings
    x_df = pd.DataFrame(x.reshape(1, -1), columns=scaler_X.feature_names_in_)
    x_scaled = scaler_X.transform(x_df)
    y_scaled = model.predict(x_scaled, verbose=0)
    y = scaler_y.inverse_transform(y_scaled)
    return y[0]

def predict_batch(x_batch):
    """Helper function to predict outputs for a batch of inputs."""
    x_df = pd.DataFrame(x_batch, columns=scaler_X.feature_names_in_)
    x_scaled = scaler_X.transform(x_df)
    y_scaled = model.predict(x_scaled, verbose=0)
    y = scaler_y.inverse_transform(y_scaled)
    return y

def objective_function(x):
    """The objective is to maximize surface area, so we minimize its negative."""
    predictions = predict_outputs(x)
    surface_area = predictions[2]
    return -surface_area

def constraint_functions(x):
    """Returns the values for the constraints."""
    pressure_drop, avg_velocity, _, mass = predict_outputs(x)
    
    # Mass < 125  => 125 - Mass > 0
    # Pressure Drop < 8000 => 8000 - Pressure Drop > 0
    # Avg Velocity > 520 => Avg Velocity - 520 > 0
    return [
        125 - mass,
        8000 - pressure_drop,
        avg_velocity - 520
    ]

## Perform Optimization
Now we can run the optimization. We'll define the bounds for the input variables and the constraints.

In [ ]:
import matplotlib.pyplot as plt

# Define bounds for the input variables
bounds = Bounds(
    [10, 10, 2500],  # [X Cell Size, YZ Cell Size, Velocity Inlet]
    [25, 25, 3500]
)

# --- Analyze Search Space ---
print("Performing random search to analyze the design space...")
num_samples = 1000000
random_samples = np.random.uniform(low=bounds.lb, high=bounds.ub, size=(num_samples, 3))

# Predict outputs for all samples in a batch
predicted_outputs = predict_batch(random_samples)

# Check which samples satisfy the constraints
pressure_drop = predicted_outputs[:, 0]
avg_velocity = predicted_outputs[:, 1]
surface_area = predicted_outputs[:, 2]
mass = predicted_outputs[:, 3]

feasible_mask = (mass < 125) & (pressure_drop < 8000) & (avg_velocity > 520)
num_feasible = np.sum(feasible_mask)

print(f"\nFound {num_feasible} feasible points out of {num_samples} random samples.")

if num_feasible > 0:
    feasible_inputs = random_samples[feasible_mask]
    feasible_outputs = predicted_outputs[feasible_mask]
    
    # Find the best point from the random search
    best_random_idx = np.argmax(feasible_outputs[:, 2]) # Index of max surface area
    best_random_input = feasible_inputs[best_random_idx]
    best_random_output = feasible_outputs[best_random_idx]

    print("\nBest point found via random search:")
    print(f"  Inputs: X Cell Size={best_random_input[0]:.2f}, YZ Cell Size={best_random_input[1]:.2f}, Velocity={best_random_input[2]:.2f}")
    print(f"  Outputs: Surface Area={best_random_output[2]:.2f}, Mass={best_random_output[3]:.2f}, Pressure Drop={best_random_output[0]:.2f}, Avg Velocity={best_random_output[1]:.2f}")

    # Visualize the feasible region
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    sc = ax.scatter(feasible_inputs[:, 0], feasible_inputs[:, 1], feasible_inputs[:, 2], c=feasible_outputs[:, 2], cmap='viridis', label='Feasible Points')
    ax.scatter(best_random_input[0], best_random_input[1], best_random_input[2], c='red', s=100, marker='*', label='Best Random Point')
    ax.set_xlabel('X Cell Size (mm)')
    ax.set_ylabel('YZ Cell Size (mm)')
    ax.set_zlabel('Inlet Velocity (mm/s)')
    ax.set_title('Feasible Design Space')
    plt.colorbar(sc, label='Surface Area')
    plt.legend()
    plt.show()
else:
    print("\nCould not find any feasible points with random sampling. The constraints may be too strict.")


In [ ]:
# Define the constraints
# All constraint function values must be non-negative
nonlinear_constraint = NonlinearConstraint(constraint_functions, 0, np.inf)

# --- Multi-start Optimization ---
num_starts = 50
results = []

print(f"Starting optimization with {num_starts} random starting points...")
start_time = time.time()

# Decide where to place the multi-start initial guesses.
# If a best point was found during the random search (best_random_input),
# place starts nearby by sampling from a normal distribution centered on it.
# Otherwise, fall back to the center of the bounds with small perturbations.
if 'best_random_input' in locals():
    center = np.asarray(best_random_input)
    print("Using best random point as center for multi-start sampling:", center)
else:
    # fallback: center of bounds
    center = (np.asarray(bounds.lb) + np.asarray(bounds.ub)) / 2.0
    print("No best random point found; using center of bounds for multi-start sampling:", center)

# Standard deviation for perturbations: choose 20% of the variable range
range_ = np.asarray(bounds.ub) - np.asarray(bounds.lb)
std = 0.1 * range_

for i in range(num_starts):
    # sample around the center with Gaussian noise, then clip to bounds
    x0 = np.random.normal(loc=center, scale=std)
    x0 = np.clip(x0, bounds.lb, bounds.ub)

    print(f"\n--- Run {i+1}/{num_starts} --- Initial guess: {x0}")
    # Run the optimization
    result = minimize(
        objective_function,
        x0,
        method='SLSQP',
        bounds=bounds,
        constraints=[nonlinear_constraint],
        options={'disp': False, 'ftol': 1e-9} # disp=False to reduce output
    )
    
    if result.success:
        results.append(result)
        print(f"Run {i+1} successful. Objective: {-result.fun:.4f}")
    else:
        print(f"Run {i+1} failed: {result.message}")


end_time = time.time()
print(f"\nOptimization finished in {end_time - start_time:.4f} seconds.")

## Results
Let's examine the results of the optimization.

In [ ]:
if results:
    # Find the best result (highest surface area -> minimum function value)
    best_result = min(results, key=lambda r: r.fun)
    
    optimal_inputs = best_result.x
    optimal_outputs = predict_outputs(optimal_inputs)
    
    print("Best optimization result found!")
    print("\nOptimal Input Parameters:")
    print(f"  X Cell Size: {optimal_inputs[0]:.4f} mm")
    print(f"  YZ Cell Size: {optimal_inputs[1]:.4f} mm")
    print(f"  Inlet Velocity: {optimal_inputs[2]:.4f} mm/s")
    
    print("\nPredicted Output at Optimal Point:")
    print(f"  Maximized Surface Area: {-best_result.fun:.4f}")
    print(f"  Pressure Drop: {optimal_outputs[0]:.4f} Pa")
    print(f"  Average Velocity: {optimal_outputs[1]:.4f} mm/s")
    print(f"  Mass: {optimal_outputs[3]:.4f} g")
else:
    print("No successful optimization runs.")


In [ ]:
import os
import subprocess
import json

# --- Validate with nTop ---
if 'best_result' in locals() and hasattr(best_result, 'success') and best_result.success:
    print("--- Validating Optimal Point with nTop Simulation ---")
    
    # Get the optimal inputs from the best result
    optimal_inputs_validation = best_result.x

    # Define paths and file names
    exe_path = r"C:/Program Files/nTopology/nTopology/nTopCL.exe"
    ntop_file_path = r"nTop/nTop_ASME_Hackathon_HEX.ntop"
    input_file_name = "validation_input.json"
    output_file_name = "validation_output.json"

    # Check if nTopCL.exe exists
    if not os.path.exists(exe_path):
        print(f"\nError: nTopCL.exe not found at '{exe_path}'")
        print("Please update the 'exe_path' variable with the correct path to your nTopCL.exe")
    else:
        # Create the JSON input for nTop, matching the structure from main.py
        inputs_json = {
            "description": "Validation run with optimal parameters",
            "inputs": [
                {
                    "description": "Optimal cell size in X",
                    "name": "Cell Size X",
                    "type": "real",
                    "units": "mm",
                    "value": optimal_inputs_validation[0]
                },
                {
                    "description": "Optimal cell size in Y/Z",
                    "name": "Cell Size Y/Z",
                    "type": "real",
                    "units": "mm",
                    "value": optimal_inputs_validation[1]
                },
                {
                    "description": "Optimal inlet velocity",
                    "name": "Inlet Velocity",
                    "type": "real",
                    "units": "mm*s^-1",
                    "value": optimal_inputs_validation[2]
                }
            ],
            "title": "Heat Exchanger Validation"
        }

        with open(input_file_name, 'w') as f:
            json.dump(inputs_json, f, indent=4)

        # Construct and run the nTop command
        arguments = [exe_path, "-j", input_file_name, "-o", output_file_name, ntop_file_path]
        print("\nRunning nTop simulation...")
        
        try:
            process = subprocess.run(arguments, capture_output=True, text=True, check=True, timeout=300)
            print("nTop simulation completed successfully.")

            # Read the output from nTop
            with open(output_file_name, 'r') as f:
                ntop_output_data = json.load(f)
            
            # Extract the relevant values
            simulated_results = ntop_output_data[0]['value']['jsonObject']

            print("\n--- Comparison of Results ---")
            print(f"{'Output':<20} | {'Model Prediction':<20} | {'nTop Simulation':<20}")
            print("-" * 65)
            
            predicted_outputs = predict_outputs(optimal_inputs_validation)
            
            print(f"{'Surface Area':<20} | {predicted_outputs[2]:<20.4f} | {simulated_results.get('Surface Area', 'N/A'):<20}")
            print(f"{'Pressure Drop':<20} | {predicted_outputs[0]:<20.4f} | {simulated_results.get('PressureDrop', 'N/A'):<20}")
            print(f"{'Avg Velocity':<20} | {predicted_outputs[1]:<20.4f} | {simulated_results.get('AvgVelocity', 'N/A'):<20}")
            print(f"{'Mass':<20} | {predicted_outputs[3]:<20.4f} | {simulated_results.get('Mass', 'N/A'):<20}")

        except FileNotFoundError:
             print(f"\nError: nTopCL.exe not found at '{exe_path}'")
             print("Please update the 'exe_path' variable with the correct path to your nTopCL.exe")
        except subprocess.CalledProcessError as e:
            print("\nError during nTop simulation:")
            print("\n--- STDOUT ---")
            print(e.stdout)
            print("\n--- STDERR ---")
            print(e.stderr)
        except subprocess.TimeoutExpired:
            print("\nError: nTop simulation timed out after 5 minutes.")
        except Exception as e:
            print(f"\nAn unexpected error occurred: {e}")

else:
    print("\nSkipping nTop validation because no successful optimization result was found.")
